# 06 — Branches: acute vs acute + pre-existing lesion features

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "analysis" / "lib").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from analysis.lib.data import (load_label, restricted_pool,
                               build_demographics_matrix, identify_binary_targets)
from analysis.lib.pipelines import process_fold
from analysis.lib.stacking import process_stacking_fold
from analysis.lib.cv import pooled_cv

DATA_DIR           = PROJECT_ROOT / "data"
LACUNA_DIR         = DATA_DIR / "lacuna" / "METAVCI_PSCI_LACUNA" / "derivatives"
LACUNA_CHRONIC_DIR = DATA_DIR / "lacuna" / "METAVCI_PSCI_CHRONIC_LACUNA" / "derivatives"
BR_OUT             = PROJECT_ROOT / "outputs" / "predictive_modeling_branches"

CLIN_PATH = DATA_DIR / "tabular" / "clinical_data.csv"
assert CLIN_PATH.exists(), f"missing {CLIN_PATH} — see data/README.md"

# branch -> (chronic label, derivatives dir, pool rule)
BRANCHES = {
    "acute_vs_wmh":        ("WMH",        LACUNA_DIR,         dict(all_of=["acuteinfarct", "WMH"])),
    "acute_vs_oldinfarct": ("oldinfarct", LACUNA_DIR,         dict(all_of=["acuteinfarct", "oldinfarct"])),
    "acute_vs_lacune":     ("lacune",     LACUNA_DIR,         dict(all_of=["acuteinfarct", "lacune"])),
    "acute_vs_chronic":    ("chronic",    LACUNA_CHRONIC_DIR, dict(all_of=["acuteinfarct"],
                                                                   any_of=["WMH", "oldinfarct", "lacune"])),
}

clinical_df = pd.read_csv(CLIN_PATH, index_col=0)
cv = pooled_cv(seed=42)

In [ ]:
# Paired stacker (arm A vs arm B) + per-modality single-view comparison
for branch, (label, deriv_dir, pool_rule) in BRANCHES.items():
    out_dir = BR_OUT / branch
    out_dir.mkdir(parents=True, exist_ok=True)
    if (out_dir / "stacker_scores.csv").exists():
        print(f"[skip] {branch}: outputs exist")
        continue

    pool = restricted_pool(LACUNA_DIR, clinical_df, **pool_rule)
    clinical = clinical_df.loc[pool]
    fill = dict(fill_value=0) if branch == "acute_vs_chronic" else {}

    lesion_acute = load_label(LACUNA_DIR, "acuteinfarct", "rd",  "damagepct").reindex(pool)
    flnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "fnm", "tmap").reindex(pool)
    slnm_acute   = load_label(LACUNA_DIR, "acuteinfarct", "snm", "disconnectionpct").reindex(pool)
    lesion_chr   = load_label(deriv_dir, label, "rd",  "damagepct").reindex(pool, **fill)
    flnm_chr     = load_label(deriv_dir, label, "fnm", "tmap").reindex(pool, **fill)
    slnm_chr     = load_label(deriv_dir, label, "snm", "disconnectionpct").reindex(pool, **fill)

    X_demo, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
    binary_targets = identify_binary_targets(clinical)
    print(f"=== {branch}: pool {len(pool)} ===")

    scores_list, coefs_list, per_mod_scores = [], [], []
    for target in binary_targets:
        y_series = clinical[target]
        valid = y_series.notna()
        y = y_series[valid].values
        X_base = X_demo[valid.values]
        n_demo = X_base.shape[1]
        splits = list(cv.split(X_base, y))

        arm_a = [
            ("baseline",     X_base),
            ("lesion_acute", np.hstack([X_base, lesion_acute.loc[valid].values])),
            ("flnm_acute",   np.hstack([X_base, flnm_acute.loc[valid].values])),
            ("slnm_acute",   np.hstack([X_base, slnm_acute.loc[valid].values])),
        ]
        arm_b = arm_a + [
            (f"lesion_{label}", np.hstack([X_base, lesion_chr.loc[valid].values])),
            (f"flnm_{label}",   np.hstack([X_base, flnm_chr.loc[valid].values])),
            (f"slnm_{label}",   np.hstack([X_base, slnm_chr.loc[valid].values])),
        ]
        print(f"--- {target} (N={len(y)}) ---")
        for arm_label, views in [("a_acute_only", arm_a), (f"b_acute_plus_{label}", arm_b)]:
            fold_results = Parallel(n_jobs=-1)(
                delayed(process_stacking_fold)(i, tr, te, views, y, n_demo)
                for i, (tr, te) in enumerate(splits))
            for r in fold_results:
                scores_list.append({
                    "Target": target, "Arm": arm_label, "Fold": r["Fold"],
                    "AUPRC": r["AUPRC"], "AUPRC_train": r["AUPRC_train"],
                    "meta_C": r["meta_C"]})
                coefs_list.append({
                    "Target": target, "Arm": arm_label, "Fold": r["Fold"],
                    **{f"coef_{lbl}": c for lbl, c in zip(r["view_labels"], r["meta_coefs"])},
                    "intercept": r["meta_intercept"]})
            print(f"  {arm_label}: {np.nanmean([r['AUPRC'] for r in fold_results]):.3f}")

        for mod, X_a_extra, X_b_extra in [
            ("Lesion", lesion_acute.loc[valid].values,
             np.hstack([lesion_acute.loc[valid].values, lesion_chr.loc[valid].values])),
            ("fLNM", flnm_acute.loc[valid].values,
             np.hstack([flnm_acute.loc[valid].values, flnm_chr.loc[valid].values])),
            ("sLNM", slnm_acute.loc[valid].values,
             np.hstack([slnm_acute.loc[valid].values, slnm_chr.loc[valid].values])),
        ]:
            X_a = np.hstack([X_base, X_a_extra])
            X_b = np.hstack([X_base, X_b_extra])
            for arm, X in [("a_acute_only", X_a), (f"b_acute_plus_{label}", X_b)]:
                folds = Parallel(n_jobs=-1)(
                    delayed(process_fold)(i, tr, te, X, y, mod, target, n_demo)
                    for i, (tr, te) in enumerate(splits))
                for r in folds:
                    r["Modality"] = mod
                    r["Arm"] = arm
                per_mod_scores.extend(folds)

    pd.DataFrame(scores_list).to_csv(out_dir / "stacker_scores.csv", index=False)
    pd.DataFrame(coefs_list).to_csv(out_dir / "stacker_meta_coefficients.csv", index=False)
    pd.DataFrame(per_mod_scores).to_csv(out_dir / "scores.csv", index=False)
    print(f"saved {branch} outputs to {out_dir}")

In [ ]:
# Baseline-only model within each branch subset (same splits), and gain
# decomposition: baseline -> acute integrated (arm A) -> + pre-existing (arm B).
from analysis.lib.stats import nadeau_bengio_test

for branch, (label, deriv_dir, pool_rule) in BRANCHES.items():
    out_dir = BR_OUT / branch
    if (out_dir / "baseline_scores.csv").exists():
        print(f"[skip] {branch}: baseline outputs exist")
        continue
    pool = restricted_pool(LACUNA_DIR, clinical_df, **pool_rule)
    clinical = clinical_df.loc[pool]
    X_demo, _, _, _, _ = build_demographics_matrix(clinical, exclude_cohort=False)
    rows = []
    for target in identify_binary_targets(clinical):
        valid = clinical[target].notna()
        y = clinical[target][valid].values
        X_base = X_demo[valid.values]
        folds = Parallel(n_jobs=-1)(
            delayed(process_fold)(i, tr, te, X_base, y, "Baseline", target, X_base.shape[1])
            for i, (tr, te) in enumerate(cv.split(X_base, y)))
        rows.extend(folds)
    pd.DataFrame(rows).to_csv(out_dir / "baseline_scores.csv", index=False)
    print(f"saved {branch} baseline scores (pool {len(pool)})")

dec = []
for branch, (label, _, _) in BRANCHES.items():
    out_dir = BR_OUT / branch
    base = pd.read_csv(out_dir / "baseline_scores.csv")
    stk = pd.read_csv(out_dir / "stacker_scores.csv")
    for target, b in base.groupby("Target"):
        b = b.set_index("Fold")["AUPRC"]
        a = stk[(stk.Target == target) & (stk.Arm == "a_acute_only")].set_index("Fold")["AUPRC"]
        p = stk[(stk.Target == target) & (stk.Arm == f"b_acute_plus_{label}")].set_index("Fold")["AUPRC"]
        dec.append({"Branch": branch, "Target": target,
                    "baseline": b.mean(), "acute": a.mean(), "acute_plus": p.mean(),
                    "gain_acute": (a - b).mean(), "gain_preexisting": (p - a).mean(),
                    "p_acute_vs_baseline": nadeau_bengio_test(
                        a.loc[b.index].values, b.values, n_splits=10, test_size=0.1,
                        alternative="greater")["p"]})
dec = pd.DataFrame(dec)
dec["ratio_preexisting_to_acute"] = dec["gain_preexisting"] / dec["gain_acute"]
dec.to_csv(BR_OUT / "gain_decomposition.csv", index=False)
print(dec.round(3).to_string(index=False))
